# IRAF Aspect-Feature Mapping

- **Step 1**  Select and define the reference (master) sequence
- **Step 2**  Map SAT and MoLab features to IRAF movement aspects per phase

In [203]:
import pandas as pd
from pathlib import Path

RQ2_DIR = Path('../data/processed/rq2')
RQ2_DIR.mkdir(parents=True, exist_ok=True)

KORS_VIDEOS = ['028', '045', '059', '074', '091']
RAK_VIDEOS  = ['030', '047', '061', '076', '093']

print('Output dir:', RQ2_DIR)
print('KORS videos:', KORS_VIDEOS)
print('RAK  videos:', RAK_VIDEOS)


Output dir: ../data/processed/rq2
KORS videos: ['028', '045', '059', '074', '091']
RAK  videos: ['030', '047', '061', '076', '093']


## Reference

SB (Fp1) is used as the reference.
- sit-to-stand: `028`
- stand-to-sit: `030`

In [204]:
reference = {
    'participant': 'SB (Fp1)',
    'sit_to_stand_video': '028',
    'stand_to_sit_video': '030'
    }
print('Reference participant:', reference['participant'])
print('  Sit-to-stand video :', reference['sit_to_stand_video'])
print('  Stand-to-sit video :', reference['stand_to_sit_video'])

Reference participant: SB (Fp1)
  Sit-to-stand video : 028
  Stand-to-sit video : 030


## Aspect Mapping

Each IRAF aspect is linked to one or more SAT or MoLab features.

- SAT is used first
- MoLab is added when SAT is not enough
- Aspects where all participants had NaN IRAF scores are excluded

In [205]:
EXCEL_RAK  = Path('../data/original/Final_IRAF_SAT_Rak_Bedömare sep och gemensamt.xlsx')
EXCEL_KORS = Path('../data/original/Forts_IRAF_SAT_Kors _Gemensamt.xlsx')
EXCEL_FILE = EXCEL_RAK

PHASE_MAPPING    = {'Utgångsposition': 'f1', 'Rörelsestart': 'f2', 'Rörelseutförande': 'f3'}
MOVEMENT_MAPPING = {'Sittande till stående': '2a_1', 'Stående till sittande': '2a_2'}

aspect_rows = []
xl = pd.ExcelFile(EXCEL_FILE)

for sheet_name, movement_id in MOVEMENT_MAPPING.items():
    if sheet_name not in xl.sheet_names:
        print(f'Sheet not found: {sheet_name}')
        continue
    df_sheet = pd.read_excel(EXCEL_FILE, sheet_name=sheet_name)
    gem_col = next((c for c in df_sheet.columns if str(c).startswith('Gem_Avvikelse')), None)
    current_phase   = None
    counter         = 0
    aspects_started = False

    for _, row in df_sheet.iterrows():
        col0 = str(row.iloc[0]).replace('\xa0', ' ').strip() if pd.notna(row.iloc[0]) else ''
        if not col0:
            continue
        if col0.lower().startswith('fp') and '_' in col0:
            if aspects_started:
                break
            continue
        if col0 in PHASE_MAPPING:
            current_phase = PHASE_MAPPING[col0]
            counter = 0
            continue
        if current_phase:
            counter += 1
            aspects_started = True
            gem_val = row[gem_col] if gem_col is not None else None
            if pd.isna(gem_val):
                continue
            aspect_rows.append({'movement': movement_id, 'phase': current_phase,
                                 'aspect_id': counter, 'aspect_name': col0})

aspects_df = pd.DataFrame(aspect_rows)
print(f'Parsed {len(aspects_df)} unique aspects from Excel')
print(aspects_df.groupby(['movement', 'phase']).size().rename('n_aspects').to_string())


Parsed 36 unique aspects from Excel
movement  phase
2a_1      f1        8
          f2        1
          f3       10
2a_2      f1        9
          f2        1
          f3        7


### Load available features

Feature from `features/` and `features_down/`


In [206]:
FEATURES_BASE = Path('../data/processed')

col_lookup = {}
for d in ['features', 'features_down']:
    for csv_path in sorted((FEATURES_BASE / d).glob('*.csv')):
        fname    = csv_path.name
        movement = '2a_2' if 'down' in fname else '2a_1'
        source   = 'MoLab' if 'molab' in fname else 'SAT'
        for col in pd.read_csv(csv_path, nrows=0).columns:
            if col.lower() in ('video_id', 'video', 'index') or col.startswith('Unnamed'):
                continue
            col_lookup[(movement, col)] = {'features_dir': d, 'csv_file': fname, 'source': source}

print(f'\nDiscovered {len(col_lookup)} feature columns across both movements')

PREFIX_CONTEXT = {
    'forb': 'förberedelsefas',
    'uppr': 'uppresningsfas',
    'fas1': 'initial lowering phase (fas 1)',
    'fas2': 'final descent phase (fas 2)',
}

def col_to_rationale(col):
    parts = col.split('_')
    prefix = parts[0].lower()
    if prefix in PREFIX_CONTEXT:
        body = ' '.join(parts[1:])
        return f"{body.replace('_', ' ').capitalize()} during {PREFIX_CONTEXT[prefix]}"
    return ' '.join(parts).replace('_', ' ').capitalize()



Discovered 115 feature columns across both movements


### Map features to aspects 

For each aspect the most relevant SAT feature(s) are selected first.
MoLab is added only when it contributes information SAT does not capture well.

In [207]:
ASPECT_FEATURES = {('2a_1', 'f1', 2): [('mean_left_knee_angle',
                      'SAT left knee flexion angle directly quantifies whether the knee exceeds the required 90° '
                      'starting position'),
                     ('mean_right_knee_angle',
                      'SAT right knee flexion angle directly quantifies whether the knee exceeds the required 90° '
                      'starting position')],
 ('2a_1', 'f1', 3): [('delta_knee_angle',
                      'SAT left-right knee angle difference indicates whether kneecaps are symmetrically directed '
                      'forward'),
                     ('delta_knee_asymmetry',
                      'MoLab left-right knee angle difference directly quantifies kneecap alignment asymmetry')],
 ('2a_1', 'f1', 4): [('mean_trunk_angle',
                      'SAT trunk sagittal angle captures pelvic orientation in the starting posture'),
                     ('mean_pelvis_tilt',
                      'MoLab pelvis tilt is added because it measures slight anterior pelvic tilt more directly than '
                      'SAT')],
 ('2a_1', 'f1', 5): [('delta_hip_angle',
                      'SAT left-right hip height difference serves as a proxy for frontal-plane pelvis asymmetry'),
                     ('mean_pelvis_lateral',
                      'MoLab lateral pelvic tilt directly measures whether crista edges are at the same level')],
 ('2a_1', 'f1', 6): [('mean_trunk_angle',
                      'SAT trunk sagittal angle reflects lumbar curvature. deviation from neutral indicates '
                      'non-neutral lordosis'),
                     ('mean_pelvis_tilt',
                      'MoLab pelvis tilt confirms lumbar lordosis through the degree of anterior pelvic tilt')],
 ('2a_1', 'f1', 7): [('mean_trunk_angle',
                      'SAT trunk angle captures overall sagittal spinal curvature including thoracic kyphosis'),
                     ('mean_head_trunk_angle',
                      'SAT head-trunk angle reflects the combined cervico-thoracic alignment relative to the trunk '
                      'axis')],
 ('2a_1', 'f1', 8): [('delta_hip_height',
                      'SAT vertical hip displacement captures how far the trunk is positioned forward over the ischial '
                      'tuberosities')],
 ('2a_1', 'f1', 10): [('mean_head_trunk_angle',
                       'SAT head-trunk angle measures whether the head is centred over the trunk longitudinal axis')],
 ('2a_1', 'f1', 11): [('mean_head_trunk_angle',
                       'SAT head-trunk angle serves as a proxy for cervical lordosis via head position relative to '
                       'trunk')],
 ('2a_1', 'f2', 1): [('delta_trunk_flexion',
                      'SAT trunk flexion change quantifies the forward trunk lean during movement start'),
                     ('delta_pelvis_tilt',
                      'MoLab pelvic tilt change is added because the aspect explicitly includes anterior pelvic tilt')],
 ('2a_1', 'f3', 1): [('forb_rms_trunk_acceleration',
                      'SAT RMS trunk acceleration in the preparation phase is used as the main proxy for movement '
                      'smoothness'),
                     ('uppr_rms_trunk_acceleration',
                      'SAT RMS trunk acceleration in the rise phase complements smoothness during the upward '
                      'movement'),
                     ('forb_rms_pelvis_acc',
                      'MoLab RMS pelvis acceleration during preparation provides a 3D measure of movement smoothness')],
 ('2a_1', 'f3', 2): [('forb_delta_trunk_flexion',
                      'SAT trunk flexion range in the preparation phase quantifies the forward lean achieved before '
                      'seat-off'),
                     ('forb_max_trunk_angular_velocity',
                      'SAT peak trunk angular velocity describes the speed of trunk forward movement during '
                      'preparation')],
 ('2a_1', 'f3', 3): [('forb_trunk_hip_correlation',
                      'SAT hip-trunk trajectory correlation measures whether pelvis and trunk move forward together as '
                      'a unit'),
                     ('forb_delta_trunk_flexion',
                      'SAT trunk forward displacement confirms trunk continues moving with the pelvis during '
                      'preparation'),
                     ('forb_delta_pelvis_tilt',
                      'MoLab pelvic tilt change confirms pelvis continues moving forward together with the trunk'),
                     ('forb_peak_pelvis_gyro',
                      'MoLab peak pelvis angular velocity captures the rotational intensity of pelvic forward movement')],
 ('2a_1', 'f3', 4): [('forb_mean_head_trunk_angle',
                      'SAT head-trunk angle measures whether the head stays aligned with the trunk longitudinal axis '
                      'during movement')],
 ('2a_1', 'f3', 5): [('forb_rms_wrist_acceleration',
                      'SAT wrist acceleration reflects arm activity. low values indicate arms follow passively without '
                      'active propulsion')],
 ('2a_1', 'f3', 6): [('forb_delta_trunk_flexion',
                      'SAT trunk flexion range quantifies whether sufficient forward reach is achieved before '
                      'seat-off'),
                     ('forb_max_trunk_angular_velocity',
                      'SAT peak trunk angular velocity captures whether trunk moves with adequate momentum forward')],
 ('2a_1', 'f3', 7): [('uppr_total_knee_extension',
                      'SAT total knee extension range measures whether the knee reaches full straightening during the '
                      'rise phase'),
                     ('uppr_total_hip_extension',
                      'SAT total hip extension range measures whether the hip reaches full straightening during the '
                      'rise phase'),
                     ('uppr_peak_shank_gyro',
                      'MoLab shank angular velocity directly measures tibial forward angular movement during rise')],
 ('2a_1', 'f3', 8): [('uppr_total_knee_extension',
                      'SAT knee extension range captures the forward-upward tibial movement required for full knee '
                      'straightening'),
                     ('uppr_peak_knee_extension_velocity',
                      'SAT peak knee extension velocity quantifies the speed of tibial forward displacement during '
                      'rise'),
                     ('uppr_peak_shank_gyro',
                      'MoLab shank angular velocity directly measures tibial forward angular movement speed')],
 ('2a_1', 'f3', 9): [('uppr_total_knee_extension',
                      'SAT knee extension range confirms full knee straightening toward neutral standing position'),
                     ('uppr_total_hip_extension',
                      'SAT hip extension range confirms full hip straightening toward neutral standing position')],
 ('2a_1', 'f3', 10): [('uppr_trunk_reextension_angle',
                       'SAT trunk re-extension angle directly quantifies how close the trunk returns to the upright '
                       'neutral standing position')],
 ('2a_1', 'f3', 11): [('forb_mean_head_trunk_angle',
                       'SAT head-trunk angle is the best available proxy for whether neck and shoulders follow the '
                       'trunk movement')],
 ('2a_1', 'f3', 12): [('forb_rms_wrist_acceleration',
                       'SAT wrist acceleration captures whether arms follow the movement passively throughout the '
                       'rise')],
 ('2a_2', 'f1', 1): [('stance_to_hip_ratio',
                      'SAT stance width normalised to hip width measures whether feet are placed parallel and '
                      'symmetrically')],
 ('2a_2', 'f1', 2): [('stance_width',
                      'SAT absolute stance width measures foot placement distance relative to hip width'),
                     ('stance_to_hip_ratio',
                      'SAT hip-normalised stance ratio confirms feet are positioned directly under the hip joints')],
 ('2a_2', 'f1', 3): [('hip_asym',
                      'SAT hip asymmetry index serves as a proxy for medial-lateral weight distribution between feet'),
                     ('mean_pelvis_lateral',
                      'MoLab lateral pelvic tilt captures medial-lateral load distribution between feet')],
 ('2a_2', 'f1', 4): [('knee_asym',
                      'SAT knee asymmetry measures left-right knee alignment. neutral alignment points kneecaps toward '
                      'middle toes'),
                     ('delta_knee_angle',
                      'SAT left-right knee angle difference captures bilateral kneecap direction asymmetry')],
 ('2a_2', 'f1', 5): [('mean_left_knee_angle',
                      'SAT left knee flexion angle confirms the required semi-flexed starting position'),
                     ('mean_right_knee_angle',
                      'SAT right knee flexion angle confirms the required semi-flexed starting position'),
                     ('mean_knee_forward',
                      'SAT mean knee forward position captures the anterior tibial inclination in the semi-flexed '
                      'stance')],
 ('2a_2', 'f1', 6): [('mean_trunk_angle',
                      'SAT trunk sagittal angle captures pelvis orientation in the starting posture'),
                     ('mean_pelvis_tilt',
                      'MoLab pelvis tilt is added because it measures slight anterior pelvic tilt more directly than '
                      'SAT')],
 ('2a_2', 'f1', 7): [('mean_pelvis_lateral',
                      'MoLab lateral pelvic tilt directly measures SIAS symmetry in the frontal plane. no SAT feature '
                      'covers this')],
 ('2a_2', 'f1', 8): [('mean_trunk_angle',
                      'SAT trunk sagittal angle reflects lumbar curvature. deviations indicate hyper- or hypolordosis'),
                     ('mean_pelvis_tilt',
                      'MoLab pelvis tilt is added because lumbar lordosis cannot be inferred reliably from SAT alone')],
 ('2a_2', 'f1', 9): [('mean_trunk_angle',
                      'SAT overall trunk sagittal angle captures thoracic spinal curvature in the sagittal plane'),
                     ('mean_head_trunk_angle',
                      'SAT head-trunk angle reflects the upper thoracic and cervical contribution to kyphosis')],
 ('2a_2', 'f1', 10): [('shoulder_asym',
                       'SAT shoulder asymmetry measures horizontal shoulder level, indicating thoracic rotation or '
                       'scoliosis'),
                      ('mean_pelvis_lateral',
                       'MoLab lateral pelvic tilt complements shoulder asymmetry to capture overall frontal-plane '
                       'symmetry')],
 ('2a_2', 'f1', 11): [('mean_head_trunk_angle',
                       'SAT head-trunk angle measures whether the head is symmetrically positioned directly above the '
                       'trunk')],
 ('2a_2', 'f1', 12): [('mean_head_trunk_angle',
                       'SAT head-trunk angle serves as an indirect measure of cervical lordosis via head position '
                       'relative to trunk')],
 ('2a_2', 'f1', 13): [('shoulder_asym',
                       'SAT shoulder asymmetry directly quantifies whether shoulders are level and symmetric')],
 ('2a_2', 'f1', 15): [('mean_trunk_angle',
                       'SAT trunk sagittal angle confirms trunk is positioned upright and aligned over pelvis'),
                      ('mean_pelvis_tilt',
                       'MoLab pelvis tilt confirms neutral pelvis position under the trunk in sagittal plane')],
 ('2a_2', 'f2', 1): [('delta_trunk_flexion',
                      'SAT trunk flexion change captures the forward lean that initiates hip and knee flexion for '
                      'sit-down'),
                     ('ROM_hip',
                      'SAT hip flexion range measures the degree of hip bending that accompanies trunk flexion'),
                     ('delta_pelvis_tilt',
                      'MoLab pelvic tilt change is added because the aspect explicitly includes pelvic flexion '
                      'mechanics')],
 ('2a_2', 'f3', 1): [('fas1_ROM_trunk',
                      'SAT trunk ROM in fas 1 quantifies the forward trunk lean during the initial descent phase'),
                     ('fas1_ROM_hip',
                      'SAT hip flexion ROM in fas 1 measures the degree of hip bending in the initial descent'),
                     ('fas1_phase_duration',
                      'SAT fas 1 duration captures the temporal control of the initial lowering phase')],
 ('2a_2', 'f3', 2): [('fas1_ROM_trunk',
                      'SAT trunk ROM confirms the total forward lean range during the continued descent'),
                     ('fas1_ROM_hip',
                      'SAT hip flexion ROM directly quantifies continued hip bending during the descent')],
 ('2a_2', 'f3', 3): [('fas1_ROM_knee',
                      'SAT knee flexion ROM measures how much the knees continue to bend and advance forward'),
                     ('fas1_mean_knee_forward',
                      'SAT mean knee forward position quantifies the horizontal knee displacement over the toes')],
 ('2a_2', 'f3', 4): [('fas1_knee_asym',
                      'SAT knee asymmetry measures whether both knees maintain symmetric forward direction in fas 1'),
                     ('fas1_knee_angle_diff',
                      'MoLab left-right knee angle difference directly quantifies kneecap alignment toward the middle '
                      'toe'),
                     ('fas1_pelvis_lateral',
                      'MoLab lateral pelvis tilt in fas 1 reflects lower-limb frontal-plane alignment')],
 ('2a_2', 'f3', 5): [('fas1_peak_pelvis_velocity_y',
                      'SAT peak vertical pelvis velocity captures the maximum downward speed. high values indicate '
                      'uncontrolled drop'),
                     ('fas1_rms_pelvis_acc_y',
                      'SAT RMS vertical pelvis acceleration quantifies smoothness of descent. high values indicate '
                      'jerkiness')],
 ('2a_2', 'f3', 6): [('fas2_pelvis_backward_displacement',
                      'SAT posterior pelvis displacement measures the backward pelvis shift in the final lowering '
                      'phase'),
                     ('fas2_pelvis_descent', 'SAT vertical pelvis descent quantifies the downward movement in fas 2'),
                     ('fas2_phase_duration',
                      'SAT fas 2 duration captures the temporal control of the final descent phase')],
 ('2a_2', 'f3', 7): [('fas2_pelvis_backward_displacement',
                      'SAT posterior pelvis displacement quantifies the controlled backward-downward pelvis '
                      'trajectory'),
                     ('fas2_pelvis_descent',
                      'SAT vertical pelvis descent confirms controlled lowering toward the seat'),
                     ('fas2_rms_pelvis_acc',
                      'MoLab pelvis acceleration is added as a compact 3D control measure for the final '
                      'backward-downward transfer')],
 ('2a_2', 'f3', 8): [('fas2_knee_asym',
                      'SAT knee asymmetry measures whether knees remain directed toward the middle toe during final '
                      'descent')],
 ('2a_2', 'f3', 9): [('fas2_rms_wrist_acceleration',
                      'SAT wrist acceleration measures whether hands and arms follow passively without active midline '
                      'crossing')]}

total_pairs = sum(len(v) for v in ASPECT_FEATURES.values())
print(f'ASPECT_FEATURES: {len(ASPECT_FEATURES)} aspects defined, {total_pairs} feature-aspect pairs')
print(f'  2a_1: {sum(1 for k in ASPECT_FEATURES if k[0]=="2a_1")} aspects')
print(f'  2a_2: {sum(1 for k in ASPECT_FEATURES if k[0]=="2a_2")} aspects')
print(f'  Note: aspects without expert ratings are excluded by the merge in the next cell')


ASPECT_FEATURES: 46 aspects defined, 88 feature-aspect pairs
  2a_1: 22 aspects
  2a_2: 24 aspects
  Note: aspects without expert ratings are excluded by the merge in the next cell


In [208]:
fa_rows = []
for (movement, phase, aspect_id), entries in ASPECT_FEATURES.items():
    for col, rationale in entries:
        info = col_lookup.get((movement, col))
        if info is None:
            print(f'WARNING: ({movement}, {col}) not in col_lookup')
            continue
        fa_rows.append({
            'movement':       movement,
            'phase':          phase,
            'aspect_id':      aspect_id,
            'features_dir':   info['features_dir'],
            'csv_file':       info['csv_file'],
            'feature_column': col,
            'source':         info['source'],
            'rationale':      rationale,
        })

fa_df      = pd.DataFrame(fa_rows)
# Inner merge: keeps only aspects that have expert ratings in aspects_df
mapping_df = fa_df.merge(aspects_df, on=['movement', 'phase', 'aspect_id'], how='inner')

MAPPING_COLS = ['movement','phase','aspect_id','aspect_name',
                'features_dir','csv_file','feature_column','source','rationale']
mapping_df = mapping_df[MAPPING_COLS]

mapping_df.to_csv(RQ2_DIR / 'aspect_feature_mapping.csv', index=False)

n_rated_sts   = mapping_df[mapping_df.movement=='2a_1'][['phase','aspect_id']].drop_duplicates().shape[0]
n_rated_stsit = mapping_df[mapping_df.movement=='2a_2'][['phase','aspect_id']].drop_duplicates().shape[0]
n_defined_sts   = sum(1 for k in ASPECT_FEATURES if k[0]=='2a_1')
n_defined_stsit = sum(1 for k in ASPECT_FEATURES if k[0]=='2a_2')

print(f'aspect_feature_mapping.csv saved  ({len(mapping_df)} rows)')
print(f'  STS   rated/mapped: {n_rated_sts}/{n_defined_sts} aspects')
print(f'  StSit rated/mapped: {n_rated_stsit}/{n_defined_stsit} aspects')
print(f'  Aspects without expert ratings are excluded (inner merge with rated aspects_df)')


aspect_feature_mapping.csv saved  (60 rows)
  STS   rated/mapped: 17/22 aspects
  StSit rated/mapped: 16/24 aspects
  Aspects without expert ratings are excluded (inner merge with rated aspects_df)


In [209]:
def compact_table(df):
    return (
        df.groupby(['phase','aspect_id','aspect_name'], sort=False)
        .agg(
            candidate_features=('feature_column', lambda x: '/ '.join(x)),
            data_source=('source', lambda x: '/ '.join(x.unique())),
            rationale=('rationale', lambda x: '/ '.join(x.unique())),
        )
        .reset_index()
        .rename(columns={'aspect_name': 'aspect'})
        [['phase','aspect','candidate_features','data_source','rationale']]
    )

sts_table   = compact_table(mapping_df[mapping_df.movement == '2a_1'])
stsit_table = compact_table(mapping_df[mapping_df.movement == '2a_2'])

print('Sit-to-Stand (2a_1) - Aspect-Feature Mapping')
display(sts_table)

print('Stand-to-Sit (2a_2) - Aspect-Feature Mapping')
display(stsit_table)

Sit-to-Stand (2a_1) - Aspect-Feature Mapping


,phase,aspect,candidate_features,data_source,rationale
0,f1,Knä flex >90gr,mean_left_knee_angle/ mean_right_knee_angle,SAT,SAT left knee flexion angle directly quantifie...
1,f1,Knäskålarna riktade rakt fram,delta_knee_angle/ delta_knee_asymmetry,SAT/ MoLab,SAT left-right knee angle difference indicates...
2,f1,Bäcken i neutralposition (lätt framåttippat),mean_trunk_angle/ mean_pelvis_tilt,SAT/ MoLab,SAT trunk sagittal angle captures pelvic orien...
3,f1,Hela bålen strax framför sittbensknölarna,delta_hip_height,SAT,SAT vertical hip displacement captures how far...
4,f1,Huvudet positionerat över bålen,mean_head_trunk_angle,SAT,SAT head-trunk angle measures whether the head...
5,f1,Nacke i neutral cervikallordos,mean_head_trunk_angle,SAT,SAT head-trunk angle serves as a proxy for cer...
6,f2,Framåttippning av bäckenet och framåtfällning ...,delta_trunk_flexion/ delta_pelvis_tilt,SAT/ MoLab,SAT trunk flexion change quantifies the forwar...
7,f3,Rörelsen utförs med jämt flöde (fluid och smooth),forb_rms_trunk_acceleration/ uppr_rms_trunk_ac...,SAT/ MoLab,SAT RMS trunk acceleration in the preparation ...
8,f3,Bäckenet fortsätter framåt tillsammans med bål...,forb_trunk_hip_correlation/ forb_delta_trunk_f...,SAT/ MoLab,SAT hip-trunk trajectory correlation measures ...
9,f3,Huvudet följer i linje med bålens längsaxel,forb_mean_head_trunk_angle,SAT,SAT head-trunk angle measures whether the head...


Stand-to-Sit (2a_2) - Aspect-Feature Mapping


,phase,aspect,candidate_features,data_source,rationale
0,f1,"Står med fötterna parallellt, riktade rakt fram",stance_to_hip_ratio,SAT,SAT stance width normalised to hip width measu...
1,f1,Står med fötterna rakt under höftlederna,stance_width/ stance_to_hip_ratio,SAT,SAT absolute stance width measures foot placem...
2,f1,Knäna neutralt semiflekterade,mean_left_knee_angle/ mean_right_knee_angle/ m...,SAT,SAT left knee flexion angle confirms the requi...
3,f1,Bäckenet neutralt framåttippat i sagittalplan,mean_trunk_angle/ mean_pelvis_tilt,SAT/ MoLab,SAT trunk sagittal angle captures pelvis orien...
4,f1,Huvudet symmetrisk position rakt ovanför bålen,mean_head_trunk_angle,SAT,SAT head-trunk angle measures whether the head...
5,f1,Cervikalrygg i neutral lordos i sagittalplan,mean_head_trunk_angle,SAT,SAT head-trunk angle serves as an indirect mea...
6,f1,Axlarna symmetriska och i samma nivå,shoulder_asym,SAT,SAT shoulder asymmetry directly quantifies whe...
7,f1,Bålen placerad rakt över bäckenet i sagittalplan,mean_trunk_angle/ mean_pelvis_tilt,SAT/ MoLab,SAT trunk sagittal angle confirms trunk is pos...
8,f2,"Bålen fälls framåt, höfter och knän böjs (flek...",delta_trunk_flexion/ ROM_hip/ delta_pelvis_tilt,SAT/ MoLab,SAT trunk flexion change captures the forward ...
9,f3,Bålen fortsätter fällas framåt och höftböjning...,fas1_ROM_trunk/ fas1_ROM_hip,SAT,SAT trunk ROM confirms the total forward lean ...


In [210]:
sts_table.to_csv(RQ2_DIR / 'sit_to_stand_aspect_mapping.csv', index=False)
stsit_table.to_csv(RQ2_DIR / 'stand_to_sit_aspect_mapping.csv', index=False)
print('Saved to', RQ2_DIR)
for f in ['aspect_feature_mapping.csv',
          'sit_to_stand_aspect_mapping.csv', 'stand_to_sit_aspect_mapping.csv']:
    print(f'  {f}')


Saved to ../data/processed/rq2
  aspect_feature_mapping.csv
  sit_to_stand_aspect_mapping.csv
  stand_to_sit_aspect_mapping.csv


### Compute scores

Normalize each feature across videos, and average them within each aspect.

In [211]:
ALL_VIDEOS   = KORS_VIDEOS + RAK_VIDEOS
feature_data = {}

for _, row in mapping_df[['features_dir', 'csv_file']].drop_duplicates().iterrows():
    path = FEATURES_BASE / row['features_dir'] / row['csv_file']
    if not path.exists():
        print(f'WARNING: file not found: {path}')
        continue
    df = pd.read_csv(path)
    id_col = next((c for c in df.columns if c.lower() in ('video', 'video_id')), None)
    if id_col:
        df = df.set_index(id_col)
        df.index = df.index.map(lambda x: str(int(float(x))).zfill(3))
    df = df.loc[df.index.intersection(ALL_VIDEOS)]
    feature_data[(row['features_dir'], row['csv_file'])] = df

all_csvs = set()
for d in ['features', 'features_down']:
    for p in sorted((FEATURES_BASE / d).glob('*.csv')):
        all_csvs.add((d, p.name))
used_csvs = set(tuple(r) for _, r in mapping_df[['features_dir','csv_file']].drop_duplicates().iterrows())
unused = all_csvs - used_csvs
if unused:
    print(f'CSVs not used in mapping: {sorted(unused)}')

print(f'Loaded {len(feature_data)} / {len(all_csvs)} feature files  ({len(ALL_VIDEOS)} videos)')
first_key = next(iter(feature_data))
print(f'Sample video_ids in {first_key[1]}: {feature_data[first_key].index.tolist()}')


Loaded 12 / 12 feature files  (10 videos)
Sample video_ids in utgångsposition.csv: ['028', '030', '045', '047', '059', '061', '074', '076', '091', '093']


In [212]:
def minmax(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return series * 0.0
    return (series - mn) / (mx - mn)

score_rows = []

for (mvmt, phase, aid), group in mapping_df.groupby(['movement', 'phase', 'aspect_id']):
    normalized = []
    for _, feat_row in group.iterrows():
        col = feat_row['feature_column']
        df  = feature_data.get((feat_row['features_dir'], feat_row['csv_file']))
        if df is None or col not in df.columns:
            print(f'WARNING: {col} not found in {feat_row["csv_file"]}')
            continue
        normalized.append(minmax(df[col]))

    if not normalized:
        continue

    aspect_score = pd.concat(normalized, axis=1).mean(axis=1)
    for video_id, score in aspect_score.items():
        score_rows.append({
            'movement':    mvmt,
            'phase':       phase,
            'aspect_id':   aid,
            'video_id':    video_id,
            'aspect_score': round(score, 4),
        })

print(f'Computed {len(score_rows)} score rows')
print(f'  aspects: {len(set((r["movement"], r["phase"], r["aspect_id"]) for r in score_rows))}')
print(f'  videos:  {len(set(r["video_id"] for r in score_rows))}')


Computed 330 score rows
  aspects: 33
  videos:  10


In [213]:
SCORE_COLS = ['movement', 'phase', 'aspect_id', 'video_id', 'aspect_score']
aspect_scores_df = pd.DataFrame(score_rows, columns=SCORE_COLS)
aspect_scores_df = aspect_scores_df.merge(
    aspects_df[['movement', 'phase', 'aspect_id', 'aspect_name']],
    on=['movement', 'phase', 'aspect_id'], how='left'
)

aspect_scores_df.to_csv(RQ2_DIR / 'aspect_scores.csv', index=False)

n_aspects = aspect_scores_df[['movement','phase','aspect_id']].drop_duplicates().shape[0]
n_videos  = aspect_scores_df['video_id'].nunique()
print(f'aspect_scores.csv saved  ({len(aspect_scores_df)} rows, {n_aspects} aspects, {n_videos} videos)')
display(aspect_scores_df.head(20))


aspect_scores.csv saved  (330 rows, 33 aspects, 10 videos)


,movement,phase,aspect_id,video_id,aspect_score,aspect_name
0,2a_1,f1,2,028,0.3038,Knä flex >90gr
1,2a_1,f1,2,030,0.4544,Knä flex >90gr
2,2a_1,f1,2,045,0.3943,Knä flex >90gr
3,2a_1,f1,2,047,0.6700,Knä flex >90gr
4,2a_1,f1,2,059,0.3350,Knä flex >90gr
5,2a_1,f1,2,061,1.0000,Knä flex >90gr
6,2a_1,f1,2,074,0.0000,Knä flex >90gr
7,2a_1,f1,2,076,0.5096,Knä flex >90gr
8,2a_1,f1,2,091,0.3027,Knä flex >90gr
9,2a_1,f1,2,093,0.7700,Knä flex >90gr


In [214]:
for mvmt, label in [('2a_1', 'sit_to_stand'), ('2a_2', 'stand_to_sit')]:
    sub = aspect_scores_df[aspect_scores_df.movement == mvmt].copy()
    pivot = sub.pivot_table(
        index=['phase', 'aspect_id', 'aspect_name'],
        columns='video_id',
        values='aspect_score'
    )
    pivot.columns.name = None
    pivot = pivot.reset_index()
    out = RQ2_DIR / f'{label}_aspect_scores_normalized.csv'
    pivot.to_csv(out, index=False)
    print(f'{out.name}  ({pivot.shape[0]} aspects x {pivot.shape[1]-3} videos)')
    display(pivot)


sit_to_stand_aspect_scores_normalized.csv  (17 aspects x 10 videos)


,phase,aspect_id,aspect_name,028,030,045,047,059,061,074,076,091,093
0,f1,2,Knä flex >90gr,0.3038,0.4544,0.3943,0.6700,0.3350,1.0000,0.0000,0.5096,0.3027,0.7700
1,f1,3,Knäskålarna riktade rakt fram,0.5251,0.3999,0.2172,0.1212,0.5000,0.7299,0.4091,0.4448,0.4437,0.5000
2,f1,4,Bäcken i neutralposition (lätt framåttippat),0.3674,0.4394,0.3674,0.4760,0.4501,0.3118,0.6348,0.4599,0.5000,0.5793
3,f1,8,Hela bålen strax framför sittbensknölarna,0.4172,0.1865,0.0000,0.0542,0.8582,0.4169,0.4342,0.3519,1.0000,0.4901
4,f1,10,Huvudet positionerat över bålen,0.5978,0.6405,0.5850,0.9618,0.4077,1.0000,0.0000,0.8982,0.7014,0.8343
5,f1,11,Nacke i neutral cervikallordos,0.5978,0.6405,0.5850,0.9618,0.4077,1.0000,0.0000,0.8982,0.7014,0.8343
6,f2,1,Framåttippning av bäckenet och framåtfällning ...,0.5403,0.6081,0.7182,0.6076,0.7554,0.3964,0.1365,0.4467,0.5124,0.3392
7,f3,1,Rörelsen utförs med jämt flöde (fluid och smooth),0.8552,0.3797,0.5774,0.1622,0.3915,0.1471,0.5995,0.3560,0.7987,0.0313
8,f3,3,Bäckenet fortsätter framåt tillsammans med bål...,0.4561,0.3050,0.5004,0.4580,0.3391,0.3129,0.4595,0.2974,0.6351,0.1073
9,f3,4,Huvudet följer i linje med bålens längsaxel,0.4945,0.2993,0.2436,0.8908,0.0437,1.0000,0.7827,0.9356,0.0000,0.5977


stand_to_sit_aspect_scores_normalized.csv  (16 aspects x 10 videos)


,phase,aspect_id,aspect_name,028,030,045,047,059,061,074,076,091,093
0,f1,1,"Står med fötterna parallellt, riktade rakt fram",0.8107,0.3967,0.3821,0.5673,0.2140,0.0706,1.0000,0.0000,0.7134,0.2185
1,f1,2,Står med fötterna rakt under höftlederna,0.7028,0.4897,0.4785,0.6918,0.1399,0.1482,1.0000,0.0486,0.7949,0.1092
2,f1,5,Knäna neutralt semiflekterade,0.3588,0.7782,0.2428,0.6141,0.3830,0.3123,0.4398,0.6948,0.7917,0.7807
3,f1,6,Bäckenet neutralt framåttippat i sagittalplan,0.0250,0.1459,0.9369,0.2535,0.5752,0.5087,0.5141,0.4470,0.4844,0.2031
4,f1,11,Huvudet symmetrisk position rakt ovanför bålen,0.3429,0.2588,0.5112,0.3475,0.4115,0.6045,1.0000,0.1696,0.3741,0.0000
5,f1,12,Cervikalrygg i neutral lordos i sagittalplan,0.3429,0.2588,0.5112,0.3475,0.4115,0.6045,1.0000,0.1696,0.3741,0.0000
6,f1,13,Axlarna symmetriska och i samma nivå,0.0000,1.0000,0.1836,0.0025,0.3783,0.9659,0.3362,0.7577,0.0442,0.5373
7,f1,15,Bålen placerad rakt över bäckenet i sagittalplan,0.0250,0.1459,0.9369,0.2535,0.5752,0.5087,0.5141,0.4470,0.4844,0.2031
8,f2,1,"Bålen fälls framåt, höfter och knän böjs (flek...",0.5560,0.4916,0.5505,0.8373,0.6946,0.2200,0.3399,0.2780,0.0455,0.2911
9,f3,2,Bålen fortsätter fällas framåt och höftböjning...,0.4257,0.6079,0.3350,1.0000,0.9487,0.2031,0.2772,0.4191,0.0646,0.3957
